# Frota estimada de sábado e domingo

`Tecnologias.csv` (pedido 096412) só traz frota de dia útil (`UTIL`). Pela lei de Little, a frota que a programação exige é proporcional às partidas/hora no pico (partidas/hora × tempo de ciclo). Supondo o **mesmo tempo de ciclo do dia útil**:

`frota_tipo_dia(linha, tecnologia) = frota_util(linha, tecnologia) × pico_tipo_dia / pico_util`

`pico` = máximo, nas horas do dia, de partidas por hora somando os dois sentidos. Linhas sem partidas de dia útil no mês ficam de fora.

**Leitura obrigatória dos resultados.** A soma de `Frota` por linha **não é a frota da cidade**: dá ~38,7 mil em dia útil contra ~13 mil de frota total oficial (SPTrans, via Diário do Transporte, jan/2025), porque um mesmo veículo opera em mais de uma linha e é contado em cada uma (hipótese, não testada). Para domingo, a SPTrans informa ~4,9 mil ônibus (25/01/2026 e 02/08/2026) contra ~20 mil aqui. O que se aproveita é a **proporção** (domingo/sábado: 0,74 aqui, ~0,71 no oficial) e a variação percentual entre anos; o nível absoluto não. Domingo/dia útil aqui é 0,52–0,54; com ~12 mil de frota operante em dia útil o oficial daria ~0,41.

In [1]:
import os
from pathlib import Path

import duckdb

RAIZ_SPTRANS = r"C:\Users\9837292\Desktop\SSD\SPTrans"
PASTA_OFERTA = os.path.join(RAIZ_SPTRANS, "pedido 096412- oferta")
ANOS = ["2023", "2024", "2025"]
DIR_TABELAS = Path("../outputs/02/tabelas")

PARTIDAS = os.path.join(PASTA_OFERTA, "Partidas.csv").replace("\\", "/")
TECNOLOGIAS = os.path.join(PASTA_OFERTA, "Tecnologias.csv").replace("\\", "/")

## 1. Pico de partidas por hora, por linha × mês × tipo de dia

In [2]:
con = duckdb.connect()
con.execute(f"""
CREATE VIEW pico AS
SELECT "Linha" AS linha, "Ano Mês" AS ano_mes, "Tipo Dia" AS tipo_dia, max(n) AS pico
FROM (
    SELECT "Linha", "Ano Mês", "Tipo Dia",
           hour(CAST("Horario" AS TIME)) + 24 * CAST("Vira" AS INTEGER) AS hora,
           count(*) AS n
    FROM read_csv('{PARTIDAS}', header=true, encoding='utf-8')
    WHERE substr(CAST("Ano Mês" AS VARCHAR), 1, 4) IN ({",".join(repr(a) for a in ANOS)})
    GROUP BY ALL
)
GROUP BY ALL
""")

## 2. Frota de dia útil por tecnologia × fator de pico de cada tipo de dia

In [3]:
linha_tec = con.execute(f"""
WITH util AS (
    SELECT "Linha" AS linha, "Ano Mês" AS ano_mes, "Tecnologia" AS tecnologia,
           sum("Frota") AS frota_util
    FROM read_csv('{TECNOLOGIAS}', header=true, encoding='utf-8')
    WHERE "Tipo Dia" = 'UTIL'
    GROUP BY ALL
)
SELECT u.linha, u.ano_mes, u.tecnologia, u.frota_util,
       p_u.pico AS pico_util, p_d.tipo_dia, p_d.pico AS pico_dia,
       u.frota_util * p_d.pico / p_u.pico AS frota_estimada
FROM util u
JOIN pico p_u ON p_u.linha = u.linha AND p_u.ano_mes = u.ano_mes AND p_u.tipo_dia = 'UTIL'
JOIN pico p_d ON p_d.linha = u.linha AND p_d.ano_mes = u.ano_mes AND p_d.tipo_dia IN ('SABD', 'DOMG')
""").df()
linha_tec["Ano"] = linha_tec["ano_mes"].astype(str).str[:4]

DIR_TABELAS.mkdir(parents=True, exist_ok=True)
linha_tec.to_parquet(DIR_TABELAS / "frota_estimada_linha_tecnologia.parquet")

## 3. Total das linhas: soma no mês, depois média dos meses do ano

In [4]:
mes = linha_tec.groupby(["Ano", "ano_mes", "tipo_dia"], as_index=False).agg(
    frota_util=("frota_util", "sum"), frota_estimada=("frota_estimada", "sum"),
    n_linhas=("linha", "nunique"))
por_tipo_dia = mes.groupby(["Ano", "tipo_dia"], as_index=False).agg(
    frota_util=("frota_util", "mean"), frota_estimada=("frota_estimada", "mean"),
    n_linhas=("n_linhas", "mean"), meses=("ano_mes", "nunique"))
por_tipo_dia["razao_vs_util"] = por_tipo_dia["frota_estimada"] / por_tipo_dia["frota_util"]
por_tipo_dia.to_parquet(DIR_TABELAS / "frota_estimada_por_tipo_dia.parquet")
por_tipo_dia.round(3)

,Ano,tipo_dia,frota_util,frota_estimada,n_linhas,meses,razao_vs_util
0,2023,DOMG,38707.50,20295.960,997.667,12,0.524
1,2023,SABD,41531.50,27595.757,1101.417,12,0.664
2,2024,DOMG,38819.75,20734.877,1000.083,12,0.534
3,2024,SABD,41774.50,28108.288,1106.333,12,0.673
4,2025,DOMG,39294.50,21122.814,1004.083,12,0.538
5,2025,SABD,42282.50,28525.368,1110.500,12,0.675
